# Exercise: Training Deep Learnign Model on Nautilus


| Component                                                      | Points |
|---------------------------------------------------------------|--------|
| Q1: Add terminal screenshot showing job completion             | 10     |
| Q2: Plot training vs testing loss curves (3 networks)          | 15     |
| Q3: Plot training vs testing accuracy curves (3 networks)      | 15     |
| Q4: Brief written report analyzing model performance and table | 15     |
| **Total**                                                     | **55** |

In this exercise, you will learn how to leverage cloud computing resources through **Kubernetes Nautilus** to build and run a deep learning model training pipeline. As part of the learning process, you will train and evaluate different **Convolutional Neural Networks (CNNs)** for image classification using the **CIFAR-10 dataset**, one of the most widely used benchmarks in computer vision. You will also practice working with **Persistent Volume Claims (PVCs)** in Kubernetes to manage data and code between shared and personal workspaces.

You may explore various CNN architectures from the following references for additional insight (optional):  
- [PyTorch Vision Models](https://docs.pytorch.org/vision/main/models.html)  
- [Dive into Deep Learning (D2L)](https://www.d2l.ai/chapter_convolutional-modern/index.html)  

For details about data sources and licensing, please refer to the `License` file included once you copy the data from our source to your PVC in the previous practice.  
If you have not yet completed the previous practice, please do so first — it is a **prerequisite** for this exercise.


## <span style='color:green'>Before you get started, ensure you have successfully complete the Practice!</span>

In [3]:
import os
import sys

from jinja2 import Template
import yaml

# Generate yml file for training jobs


Instead of maually creating multiple yml files that vary very little from each other, we can exploit this advantage by generating them automatically.

In [19]:
template = """
apiVersion: batch/v1
kind: Job
metadata:
  name: job-{{sso}}-{{netwrk2}}-train
spec:
  template:
    spec:
      automountServiceAccountToken: false
      containers:
      - name: pod-{{sso}}-{{netwrk2}}-train
        image: gitlab-registry.nrp-nautilus.io/skhan/dl_pytorch:4bd8b3be
        tty: true
        stdin: true
        workingDir: /data
        command: ["/bin/sh","-c"]
        args: 
        - python3 main.py --net {{netwrk}};
          
          
        volumeMounts:
        - name: {{persistentVolume_name}}
          mountPath: /data
        resources:
            limits:
              memory: 20Gi
              cpu: 2
              nvidia.com/gpu: 1
            requests:
              memory: 20Gi
              cpu: 2
              nvidia.com/gpu: 1
      volumes:
      - name: {{persistentVolume_name}} 
        persistentVolumeClaim:
            claimName: {{persistentVolume_name}}
      restartPolicy: Never      
  backoffLimit: 0


"""


In [20]:
j2_template = Template(template)

# Fill the variables below with the appropriate information


In [21]:
sso                   = "sknnh"     # as string, replace with your SSO
persistentVolume_name = "cloudcomp-solaiman"  # as string, replace with your pvc

In [22]:
net_list = ['ResNet18', 'PreActResNet18', 'GoogLeNet', 'DenseNet121', 'ResNeXt29_2x64d', 'MobileNet',
            'MobileNetV2', 'DPN92', 'ShuffleNetG2', 'SENet18', 'EfficientNetB0', 'RegNetX_200MF', 'SimpleDLA']

# Select any three neural networks from the above list 
selected_nets = []


In [23]:
for netwrk in selected_nets:
    data = {
        "netwrk": netwrk,
        "netwrk2": netwrk.lower(),
        "sso": sso,
        "persistentVolume_name": persistentVolume_name
    }
    output_file = j2_template.render(data)
    fileout = open("job-{}-{}-train.yml".format(sso, netwrk), "w")
    fileout.write(output_file)
    fileout.close()


### Before you continue, verify that you have created the 3 YAML files.

![Kubernetes_Exercise_CreatedYAML_FromTemplate.png MISSING](../images/Kubernetes_Exercise_CreatedYAML_FromTemplate_sk.png)

#### If so, take a moment to add these YAML files into your submission using git; create a specific commit for the _M5 Exercise YAML files_


---

# Submit jobs for training

You will have to submit each job manually, so you will have to do it 3 times since we have 3 network models. Navigate to the exercise directory. 

```bash
cd module4/exercises
```
Note, the below command is an example, where `{sso}` and `{network}` would be replaced with your actual SSO id and network as you entered for the variable above. Alternatively, you may directly copy from your YAML file names. 

```BASH
kubectl create -f job-{sso}-{network}-train.yml

# One example 
kubectl create -f job-sknnh-ResNet18-train.yml
```


You can monitor the progress of your training by checking on your jobs using `kubectl get jobs`. 

```BASH
jovyan@jupyter-sknnh-missouri-edu---62fbc201:~/module4/exercises$ kubectl get jobs
NAME                        STATUS    COMPLETIONS   DURATION   AGE
job-sknnh-googlenet-train   Running   0/1           2m39s      2m39s
job-sknnh-mobilenet-train   Running   0/1           4s         4s
job-sknnh-resnet18-train    Running   0/1           20m        20m

```

You can also monitor the progress of your training by checking on your pods using `kubectl get pods`. 

```BASH
jovyan@jupyter-sknnh-missouri-edu---62fbc201:~/module4/exercises$ kubectl get pods
NAME                              READY   STATUS    RESTARTS   AGE
job-sknnh-googlenet-train-5htck   1/1     Running   0          6m
job-sknnh-mobilenet-train-mdj98   1/1     Running   0          3m25s
job-sknnh-resnet18-train-mrcb5    1/1     Running   0          23m
```

**It may take 30 minutes to 120 minutes to complete each job, depending on the network model you choose and the computing resources you have. Be patient!**

Eventually, your pods should all be completed!

```BASH
jovyan@jupyter-sknnh-missouri-edu---62fbc201:~/module4/exercises$ kubectl get pods
NAME                              READY   STATUS      RESTARTS   AGE
job-sknnh-googlenet-train-5htck   1/1     Completed   0          57m
job-sknnh-mobilenet-train-mdj98   0/1     Completed   0          21m
job-sknnh-resnet18-train-mrcb5    0/1     Completed   0          41m
```

### If not all your jobs complete successfully, you may need to resubmit them!

Connecting to your PVC, you can check for the existence of the expected output files.
    
## Q1: Add a screenshot from the terminal
**(10 points)**

Take a screenshot from the terminal, save it as `job_completion.png`, and upload it here ([module4/exercises/](./) folder). For full points, please ensure the image is linked in the markup below.

![Your job_completion.png is MISSING](./job_completion.png)

When you have all your training files, you are good to proceed. The loss and accuracy matrices are saved in the `results` directory. Check it. 

```BASH
root@pod-name-sknnh:/data# cd results
root@pod-name-sknnh:/data/results# ls
metrics_GoogLeNet.csv  
metrics_MobileNet.csv  
metrics_ResNet18.csv
```
In addition to that, trained models are saved in the `checkpoints` folder named by networks.    
The code is set to save the weights of a model after each training. 

```bash
root@pod-name-sknnh:/data/results# cd ..
root@pod-name-sknnh:/data# cd checkpoint 
root@pod-name-sknnh:/data/checkpoint# ls
ckpt_GoogLeNet.pth  
ckpt_MobileNet.pth  
ckpt_ResNet18.pth
```


---

# Training curves


Use the 3 **metrics** csv files in the next two sections

Please consult the [Practice](../practices/PersistentVolume.ipynb) to copy the files from the persistent storage to your local Jupyter Environment.


## Q2: Plotting training vs testing loss curves for each network
Plot the training loss and testing loss for each training network in a single figure.
You will create three images, filename pattern `train_test_loss_{N}.png` where {N} is a different network that you used. 

**(15 pts)** 

In [ ]:
# Add code to read csv file of matrics 





In [ ]:
# Add code for generating images here





After uploading the generated figure, make sure you re-run the following cell to attach the images.

![Your training vs testing loss from network 1 is MISSING](./train_test_loss_N1.png)

![Your training vs testing loss from network 2 is MISSING](./train_test_loss_N2.png)

![Your training vs testing loss from network 3 is MISSING](./train_test_loss_N3.png)


## Q3: Plotting training vs testing accuracy curves for each network
Plot the training accuracy and testing accurary for each training network in a single figure.
You will create three images, filename pattern `train_test_accuracy_{N}.png` where {N} is a different network that you used. 

**(15 pts)** 

In [ ]:
# Add code for generating images here






After uploading the generated figure, make sure you re-run the following cell to attach the images.

![Your training vs testing accuracy from network 1 is MISSING](./train_test_accuracy_N1.png)

![Your training vs testing accuracy from network 2 is MISSING](./train_test_accuracy_N2.png)

![Your training vs testing accuracy from network 3 is MISSING](./train_test_accuracy_N3.png)


## Q4: Explore the generated images and write a brief report on model training 
Make sure to address the following issues in your report. Using your curves and metrics:
1. Per-network takeaway: Summarize how loss/accuracy evolves across the epochs for each network and whether the model converges smoothly or shows noise/instability.
2. Generalization gap: Report the train–test gap at the best test epoch for each network (one number for loss and one for accuracy). Note any signs of overfitting or underfitting.
3. Best performer: Identify which network achieves the best test accuracy (and lowest test loss if different). 

**(15 pts)**

**Assisting Summary Table:** Fill in the following table to assist your report writing (required) 

| Network            | Best Epoch | Train Acc | Train Loss | Test Acc | Test Loss |
|--------------------|------------|-----------|------------|----------|-----------|
| {NETWORK_NAME_1}   | {be1}      | {ta1}%    | {tl1}      | {va1}%   | {vl1}     |
| {NETWORK_NAME_2}   | {be2}      | {ta2}%    | {tl2}      | {va2}%   | {vl2}     |
| {NETWORK_NAME_3}   | {be3}      | {ta3}%    | {tl3}      | {va3}%   | {vl3}     |

# Save your Notebook, add and commit all the Artifacts, and push your work!


# Congratulations! 